In [1]:
import kagglehub
path = kagglehub.dataset_download("kingburrito666/shakespeare-plays")

100%|██████████| 4.55M/4.55M [00:01<00:00, 3.29MB/s]

Extracting files...


In [2]:
path

'/root/.cache/kagglehub/datasets/kingburrito666/shakespeare-plays/versions/4'

In [3]:
import pandas as pd
import os

# List contents of the directory to find the CSV file
print("Files in directory:", os.listdir(path))

# Automatically find the first CSV file in the directory
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
if csv_files:
    csv_path = os.path.join(path, csv_files[0])
    df = pd.read_csv(csv_path)
    print(f"Successfully loaded: {csv_files[0]}")
else:
    print("No CSV file found in the directory.")

Files in directory: ['william-shakespeare-black-silhouette.jpg', 'alllines.txt', 'Shakespeare_data.csv']
Successfully loaded: Shakespeare_data.csv


In [4]:
df.head()

,Dataline,Play,PlayerLinenumber,ActSceneLine,Player,PlayerLine
0,1,Henry IV,NaN,NaN,NaN,ACT I
1,2,Henry IV,NaN,NaN,NaN,SCENE I. London. The palace.
2,3,Henry IV,NaN,NaN,NaN,"Enter KING HENRY, LORD JOHN OF LANCASTER, the ..."
3,4,Henry IV,1.0,1.1.1,KING HENRY IV,"So shaken as we are, so wan with care,"
4,5,Henry IV,1.0,1.1.2,KING HENRY IV,"Find we a time for frighted peace to pant,"


In [5]:
# Join all the lines of datatset
text = ' '.join(df['PlayerLine'][:500])

In [6]:
text

"ACT I SCENE I. London. The palace. Enter KING HENRY, LORD JOHN OF LANCASTER, the EARL of WESTMORELAND, SIR WALTER BLUNT, and others So shaken as we are, so wan with care, Find we a time for frighted peace to pant, And breathe short-winded accents of new broils To be commenced in strands afar remote. No more the thirsty entrance of this soil Shall daub her lips with her own children's blood, Nor more shall trenching war channel her fields, Nor bruise her flowerets with the armed hoofs Of hostile paces: those opposed eyes, Which, like the meteors of a troubled heaven, All of one nature, of one substance bred, Did lately meet in the intestine shock And furious close of civil butchery Shall now, in mutual well-beseeming ranks, March all one way and be no more opposed Against acquaintance, kindred and allies: The edge of war, like an ill-sheathed knife, No more shall cut his master. Therefore, friends, As far as to the sepulchre of Christ, Whose soldier now, under whose blessed cross We ar

In [7]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

In [8]:
tokenizer=TextVectorization(
    max_tokens=1000,
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    ngrams=None,
    output_mode='int',
    output_sequence_length=None
)
tokenizer.adapt([text])
print(len(tokenizer.get_vocabulary()))

1000


In [9]:
tokenizer.get_vocabulary()[:20]

['',
 '[UNK]',
 np.str_('the'),
 np.str_('and'),
 np.str_('of'),
 np.str_('i'),
 np.str_('a'),
 np.str_('to'),
 np.str_('my'),
 np.str_('for'),
 np.str_('in'),
 np.str_('he'),
 np.str_('with'),
 np.str_('that'),
 np.str_('as'),
 np.str_('be'),
 np.str_('thou'),
 np.str_('not'),
 np.str_('will'),
 np.str_('you')]

In [10]:
input_sequence = []
for sentence in text.split(","):
    tokenized_sentence = tokenizer(sentence)

    # Convert the full tensor to a numpy array first
    tokenized_array = tokenized_sentence.numpy()

    # Extract slices as normal lists directly in the loop
    for i in range(1, len(tokenized_array)):
        input_sequence.append(tokenized_array[:i+1].tolist())

In [ ]:
input_sequence

In [12]:
max(len(x) for x in input_sequence)

59

In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [14]:
padded_input_sequence = pad_sequences(input_sequence, maxlen=59, padding='pre')

In [15]:
padded_input_sequence

array([[  0,   0,   0, ...,   0,   1,   5],
       [  0,   0,   0, ...,   1,   5, 185],
       [  0,   0,   0, ...,   5, 185,   5],
       ...,
       [  0,   0,   0, ...,   0,  46,   2],
       [  0,   0,   0, ...,  46,   2, 216],
       [  0,   0,   0, ...,   2, 216, 292]], shape=(3440, 59), dtype=int32)

In [16]:
X=padded_input_sequence[:,:-1]
y=padded_input_sequence[:,-1]

In [17]:
X.shape

(3440, 58)

In [18]:
y.shape

(3440,)

### Now it become a classification problem

In [19]:
from tensorflow.keras.utils import to_categorical
y=to_categorical(y,num_classes=1001)

In [20]:
y.shape

(3440, 1001)

In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Input

In [22]:
model=Sequential()
model.add(Input(shape=(58,)))
model.add(Embedding(1001,100))
model.add(LSTM(128,return_sequences=False))
model.add(Dense(1001,activation='softmax'))

In [23]:
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [24]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 58, 100)        │       100,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1001)           │       129,129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 346,477 (1.32 MB)

 Trainable params: 346,477 (1.32 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
model.fit( X,y,epochs=100)

Epoch 1/100
108/108 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.0637 - loss: 6.2206
Epoch 2/100
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.0657 - loss: 5.8019
Epoch 3/100
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.0672 - loss: 5.7046
Epoch 4/100
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.0779 - loss: 5.6285
Epoch 5/100
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.0828 - loss: 5.5297
Epoch 6/100
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.0927 - loss: 5.4232
Epoch 7/100
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.1052 - loss: 5.3072
Epoch 8/100
108/108 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.1169 - loss: 5.1889
Epoch 9/100
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.1276 - loss: 5.0729
Epoch 10/100
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1355 - loss: 4.9539
Epoch 11/100
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.1363 - loss: 4.8356
Epoch 12/100
108/108 ━━━━━━━━━━━━━━━━

In [28]:
import numpy as np

In [38]:
text="Find we a time for"
# tokenize

# predict
np.argmax(model.predict(tokenized_text))
for i in range(5):
  tokenized_text=tokenizer(text)
  # convert to numpy array
  tokenized_text=tokenized_text.numpy()
  # padding
  tokenized_text=pad_sequences([tokenized_text],maxlen=58,padding='pre')
  next=tokenizer.get_vocabulary()[np.argmax(model.predict(tokenized_text))]
  text=text+" "+next
  print(text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Find we a time for frighted
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Find we a time for frighted peace
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Find we a time for frighted peace to
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Find we a time for frighted peace to pant
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Find we a time for frighted peace to pant him


In [37]:
# get word from tokenizer
tokenizer.get_vocabulary()[np.argmax(model.predict(tokenized_text))]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step


np.str_('frighted')